# 实验 3：数据增强消融实验

**目的**：检验数据增强对自监督伪异常模型 CutPaste 的作用（计划书 5. 实验设计 - 实验 3）。

**变量**：训练集有无数据增强（随机水平翻转 + ±5° 旋转 + 亮度扰动）
**对象**：CutPaste 训练「有增强 / 无增强」两个版本

**输出**：`results/exp3_augmentation.csv`、`.md`

**运行提示**：权重缓存在 `checkpoints/cutpaste_*.pth`；`RETRAIN = True` 重训 2 组配置。

## ① 环境准备

In [ ]:
import os
import sys
import inspect
from pathlib import Path

# 必须先于 numpy/torch 导入设置（Windows：防 OpenBLAS 内存报错、GBK 编码报错）
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")


def _find_project_root() -> Path:
    """定位项目根目录（包含 src/ 与 configs/ 的 defect_detection 目录）。"""
    # 方式 1：由 Notebook 文件所在位置向上找（VS Code / JupyterLab 均适用，不依赖启动目录）
    for var in ("__vsc_ipynb_file__", "__file__"):
        p = globals().get(var)
        if p and str(p).endswith(".ipynb"):
            for parent in Path(p).resolve().parents:
                if (parent / "src").exists() and (parent / "configs").exists():
                    return parent
    # 方式 2：从当前工作目录向上找（经典 Jupyter Notebook 场景）
    d = Path.cwd().resolve()
    for parent in (d, *d.parents):
        if (parent / "src").exists() and (parent / "configs").exists():
            return parent
    raise FileNotFoundError(
        "未找到项目根目录（defect_detection，含 src/ 与 configs/）。\n"
        "请确认 Notebook 位于项目的 experiments/ 目录下，或从项目根目录启动 Jupyter。"
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
print("项目根目录:", PROJECT_ROOT)
print("CUDA 可用:", torch.cuda.is_available(), "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## ② 实验配置与数据准备

In [ ]:
# ============ 训练开关 ============
RETRAIN = False   # True = 忽略已有权重、从头训练（耗时数小时）；False = 复用 checkpoints 仅评估

import json

import pandas as pd

from src.experiment_utils import get_or_train, load_config, release_memory, resolve
from src.data import get_test_loader, get_train_loader
from src.utils import set_seed

cfg = load_config()
set_seed(cfg["dataset"]["seed"])
data_root = resolve(cfg["paths"]["data_root"])
results_dir = resolve(cfg["paths"]["results_dir"])

print("数据根目录:", data_root)
print("图像尺寸:", cfg["dataset"]["image_size"], "| batch_size:", cfg["dataset"]["batch_size"])
print("本实验涉及模型的超参数（configs/experiment.yaml）:")
print(json.dumps(cfg["models"], indent=2, ensure_ascii=False))

# 测试集加载器（评估用，固定不变）
# Windows Jupyter 下多进程 DataLoader 不稳，统一 num_workers=0（不影响结果，仅加载略慢）
test_loader = get_test_loader(data_root, batch_size=cfg["dataset"]["batch_size"],
                              num_workers=0, image_size=cfg["dataset"]["image_size"])

## ③ 消融实验：CutPaste 有/无增强 串行训练 + 评估

In [ ]:
from src.evaluation import evaluate_model
from src.models import build_model

results = []
for model_name in ("cutpaste",):
    for use_aug, tag in ((True, "有增强"), (False, "无增强")):
        print(f"\n{'=' * 60}\n>>> {model_name} {tag}\n{'=' * 60}")
        train_loader = get_train_loader(data_root, batch_size=cfg["dataset"]["batch_size"],
                                        num_workers=0, image_size=cfg["dataset"]["image_size"],
                                        use_augmentation=use_aug)   # ← 实验开关：数据增强
        model = build_model(model_name, device="auto",
                            image_size=cfg["dataset"]["image_size"],
                            **cfg["models"].get(model_name, {}))
        model = get_or_train(model, f"{model_name}_{tag}.pth", train_loader,
                             retrain=RETRAIN)
        res = evaluate_model(model, test_loader)
        res["model"] = f"{model_name}({tag})"
        results.append(res)
        release_memory(model)

df = pd.DataFrame(results)
df.to_csv(results_dir / "exp3_augmentation.csv", index=False, encoding="utf-8-sig")
with open(results_dir / "exp3_augmentation.md", "w", encoding="utf-8") as f:
    f.write(df.to_markdown(index=False))
df

## ④ 训练源码速览（本实验模型的 train 实现）

以下用 `inspect` 直接打印项目 `src/` 中的训练代码，与上面实际执行的训练逻辑完全一致（单一来源，无需手工复制）：
- `get_or_train`：统一训练入口（权重缓存判断 / 训练调度 / 保存）
- 各模型 `train`：本实验的训练循环实现

In [ ]:
print(inspect.getsource(get_or_train))

from src.models.cutpaste import CutPasteModel, cutpaste_batch

print(inspect.getsource(cutpaste_batch))          # CutPaste 伪异常生成（训练期数据增强核心）
print(inspect.getsource(CutPasteModel.train))     # 伪异常二分类训练

## 输出文件

`results/exp3_augmentation.csv`（增强消融对比表）。